# Patent phrase matching: starter notebook

This notebook is the **minimum working pipeline** for the mini project described in [project_brief.md](project_brief.md). It fine-tunes a pretrained Transformer **cross-encoder** to predict how similar two patent phrases are, checks it on a validation split, and writes `submission.csv`.

It is a starting point, not a solution. The baseline comparison, the experiments, and the error analysis in the brief are your work.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the output of [download_weights.ipynb](download_weights.ipynb), the saved pretrained checkpoint.
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Writes to `/kaggle/working/`: `submission.csv` and `val_predictions.csv`. A rerun overwrites both. No checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

## 1. Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [ ]:
MODEL_ID = "microsoft/deberta-v3-small"   # Hub name; the local copy is found by its last path component
MAX_LEN = 64            # anchor + target + context fit easily within 64 tokens
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2      # share of *anchors* (not rows) held out for validation
OUT_DIR = "/kaggle/working"


def find_one(pattern):
    """Return the directory of the first file under /kaggle/input that matches the pattern."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv") or find_one("train.csv")
assert DATA_DIR, "Competition data not found: add it with Add Input -> Competitions."
MODEL_PATH = find_one(f"{MODEL_ID.split('/')[-1]}/config.json") or MODEL_ID
print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "(Hub download: needs internet)")

## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [ ]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [ ]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [ ]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

## Your work starts here

Add your own sections below, or in a copy of this notebook, following the tasks in the [project brief](project_brief.md):

1. **Baseline without fine-tuning**, evaluated on the same validation split.
2. **At least two controlled experiments**, each changing one setting from Section 2 or Section 4.
3. **Error analysis** using `val_predictions.csv`.
4. **Late submission** of your final notebook, then record its public and private leaderboard scores.

Before you submit, make sure the notebook runs from top to bottom with internet off. Delete or skip experiment cells the final submission does not need, so it stays within the competition's runtime limit.